<a href="https://colab.research.google.com/github/jvit04/CursoIntroIA/blob/main/Transferencia_de_Aprendizaje.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Transferencia de Aprendizaje / Transfer Learning

In [ ]:
#Crear nuestro propio conjunto de datos

In [ ]:
!unzip babuino/babuino.zip -d babuino

In [ ]:
!unzip jirafa/jirafa.zip -d jirafa

In [ ]:
#Crear un set de datos (ya no en memoria)
!mkdir dataset
!cp -r babuino/babuino dataset/babuino
!cp -r jirafa/jirafa dataset/jirafa

In [ ]:
from tensorflow import data
#Aumento de datos
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import numpy as np

#crear el dataset generador
datagen =  ImageDataGenerator(
rescale = 1.0/255,
rotation_range = 10,
width_shift_range = 0.15,
height_shift_range = 0.15,
shear_range = 5,
zoom_range = [0.7,1.3],
validation_split = 0.2
)

data_gen_train = datagen.flow_from_directory("/content/dataset",
                                             target_size = (224,224),
                                             batch_size = 32,
                                            shuffle=True,
                                             subset = "training")
data_gen_valid = datagen.flow_from_directory("/content/dataset",
                                             target_size = (224,224),
                                             batch_size = 32,
                                            shuffle=True,
                                             subset = "validation")

In [ ]:
import matplotlib.pyplot as plt

for imagen,etiqueta in data_gen_train:
  for i in range(10):
    plt.subplot(2,5,i+1)
    plt.imshow(imagen[i])
  break
plt.show()

In [ ]:
import tensorflow as tf
import tensorflow_hub as hub

url="https://www.kaggle.com/models/google/mobilenet-v2/TensorFlow2/035-128-classification/2"
mobilenetv2 =hub.KerasLayer(url,input_shape=(224,224,3))

In [ ]:
#Importante
#Congelar las capas
mobilenetv2.trainable=False

In [ ]:
modelo = tf.keras.Sequential([
    mobilenetv2,
    tf.keras.layers.Dense(2,activation="softmax")
])

In [ ]:
modelo.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
EPOCAS = 20
train = modelo.fit(
    data_gen_train,
    epochs=EPOCAS,
    validation_data=data_gen_valid
)


In [ ]:
from PIL import Image
import cv2
import numpy as np

def categorizar(ruta):
  img=Image.open(ruta)
  img=img.resize((224,224))
  img=np.array(img)
  img=img/255
  img=img.convert("RGB")

  img = cv2.resize(img,(224,224))
  prediccion = modelo.predict(img.reshape(-1,224,224,3))
  return np.argmax(prediccion[0],axis=-1)

In [ ]:
ruta = "jirafa.jpeg"
prediccion = categorizar(ruta)
print(prediccion)

In [ ]:
ruta = "babuino.jpeg"
prediccion = categorizar(ruta)
print(prediccion)